# Day 4：SentencePiece / Unigram / tiktoken 对比 + Embedding 层设计

> **目标**：理解 Unigram 与 BPE 的差异，掌握 SentencePiece 和 tiktoken 的使用，深入理解 Embedding 层设计与词表扩展。

---

## 一、Tokenizer 技术路线全景

Day 3 我们手写了 BPE，它的核心思路是「自底向上」——从字符开始不断合并。今天补全另一条路线：

| 算法 | 方向 | 核心思想 | 代表用户 |
|------|------|---------|----------|
| **BPE** | 自底向上（合并） | 从最小单元出发，反复合并最高频 pair | GPT-2/3/4, LLaMA |
| **Unigram** | 自顶向下（删除） | 从大词表出发，逐步删除对似然贡献最小的 token | T5, ALBERT |
| **WordPiece** | 自底向上（合并） | 类似 BPE，但用似然增益（而非频率）选择合并 | BERT |

### 关键对比：BPE vs Unigram

| 维度 | BPE | Unigram |
|------|-----|--------|
| 构建方向 | 合并（频率驱动） | 裁剪（概率驱动） |
| 分词结果 | **确定性**：相同输入总得到相同分词 | **概率性**：可采样多种分词方式 |
| 数学基础 | 纯频率统计 | 一元语言模型（Unigram LM） |
| 正则化效果 | 无 | 天然的数据增强（一句话有多种分词） |
| 实现复杂度 | 简单 | 需要 EM 算法 |

## 二、Unigram 算法原理

### 2.1 核心思想

Unigram 模型假设每个 token 独立生成（一元语言模型）：

$$P(\mathbf{x}) = \prod_{i=1}^{M} P(x_i)$$

其中 $\mathbf{x} = (x_1, \ldots, x_M)$ 是文本的一种分词方式。

### 2.2 训练过程

1. **初始化**：准备一个很大的种子词表（比如所有出现过的子串，通常 100K~1M）
2. **E 步**：用 Viterbi 算法找到每句话在当前模型下的最优分词
3. **M 步**：统计每个 token 的使用频率，更新概率 $P(x_i)$
4. **裁剪**：计算每个 token 被移除后对总似然的影响，删除影响最小的 token
5. 重复 2-4 直到词表缩减到目标大小

### 2.3 裁剪标准

对于每个 token $x$，计算其被移除后的似然损失：

$$\text{loss}(x) = \sum_{s \in D} \left[ \log P(s \mid V) - \log P(s \mid V \setminus \{x\}) \right]$$

移除对似然损失最小的 token —— 即那些容易被其他 token 组合替代的 token。

### 2.4 Unigram 的独特优势：子词正则化

BPE 对同一句话只有一种分词结果，但 Unigram 可以采样多种：

```
"unaffable" 可能的分词：
  - ["un", "aff", "able"]     P = 0.4
  - ["un", "af", "f", "able"] P = 0.3
  - ["una", "ff", "able"]     P = 0.2
  ...
```

训练时随机采样不同分词 → 天然的数据增强，类似 Dropout 的正则化效果。

## 三、SentencePiece：统一的 Tokenizer 框架

SentencePiece (Kudo & Richardson, 2018) 是一个**独立于语言的** tokenizer 框架：

| 特点 | 说明 |
|------|------|
| 语言无关 | 直接处理原始 Unicode 文本，不需要预分词（如空格切分） |
| 支持两种算法 | BPE 和 Unigram 都可以 |
| 对中文友好 | 不依赖空格，天然支持中日韩等语言 |
| 可逆解码 | 编码→解码恢复原文（包括空格），空格被编码为 `▁`（U+2581） |
| 工业级使用 | LLaMA, T5, ALBERT 等主流模型都使用 |

### SentencePiece 中空格的处理

```
传统做法：先按空格切分 → 再做 BPE
  "I love NLP" → ["I", "love", "NLP"] → BPE
  问题：空格信息丢失，且依赖语言的预分词规则

SentencePiece：将空格替换为特殊字符 ▁
  "I love NLP" → "▁I▁love▁NLP" → BPE/Unigram
  结果可能：["▁I", "▁love", "▁N", "LP"]
  优势：完全可逆，语言无关
```

In [ ]:
# 安装: pip install sentencepiece
import sentencepiece as spm
import os

# 准备训练语料
corpus_text = """Natural language processing is a subfield of artificial intelligence.
Large language models have transformed how we interact with computers.
Transformers use self-attention mechanisms to process sequences efficiently.
The attention mechanism computes weighted sums of value vectors.
Byte pair encoding is the most popular tokenization method for modern LLMs.
Scaling laws predict model performance based on compute and data.
GPT models use decoder-only transformer architecture.
BERT uses bidirectional attention for language understanding tasks.
LLaMA is an open-source large language model from Meta.
DeepSeek introduced mixture of experts for efficient training.
大语言模型通过预测下一个词来学习语言规律。
注意力机制是Transformer的核心组件。
字节对编码是现代语言模型最常用的分词方法。
"""

corpus_path = "/tmp/sp_corpus.txt"
with open(corpus_path, 'w') as f:
    f.write(corpus_text)

# 训练 BPE 模式的 SentencePiece
spm.SentencePieceTrainer.train(
    input=corpus_path,
    model_prefix='/tmp/sp_bpe',
    vocab_size=200,
    model_type='bpe',
    character_coverage=0.9995,
)

# 训练 Unigram 模式的 SentencePiece
spm.SentencePieceTrainer.train(
    input=corpus_path,
    model_prefix='/tmp/sp_unigram',
    vocab_size=200,
    model_type='unigram',
    character_coverage=0.9995,
)

print("SentencePiece 训练完成!")

In [ ]:
# 加载并对比 BPE vs Unigram
sp_bpe = spm.SentencePieceProcessor(model_file='/tmp/sp_bpe.model')
sp_uni = spm.SentencePieceProcessor(model_file='/tmp/sp_unigram.model')

test_texts = [
    "Transformers are powerful models.",
    "大语言模型的注意力机制",
    "The GPT architecture uses causal attention.",
]

for text in test_texts:
    bpe_tokens = sp_bpe.encode(text, out_type=str)
    uni_tokens = sp_uni.encode(text, out_type=str)
    print(f"原文: '{text}'")
    print(f"  BPE:     {bpe_tokens} ({len(bpe_tokens)} tokens)")
    print(f"  Unigram: {uni_tokens} ({len(uni_tokens)} tokens)")
    print()

In [ ]:
# Unigram 的子词正则化：同一句话可以得到不同分词
text = "language models are transforming AI"
print(f"原文: '{text}'")
print(f"\n确定性分词 (Viterbi 最优):")
print(f"  {sp_uni.encode(text, out_type=str)}")

print(f"\n采样分词 (nbest_size=-1, alpha=0.5):")
for i in range(5):
    sampled = sp_uni.encode(text, out_type=str, enable_sampling=True, alpha=0.5, nbest_size=-1)
    print(f"  #{i+1}: {sampled}")

print("\n→ 每次采样可能不同，这就是 subword regularization 的正则化效果")

## 四、tiktoken 深入实践

tiktoken 是 OpenAI 开源的高性能 BPE tokenizer，用 Rust 实现，速度远快于 Python 版本。

### OpenAI 模型使用的 tokenizer 版本

| 编码器名称 | 使用模型 | 词表大小 |
|-----------|---------|----------|
| `gpt2` | GPT-2 | 50,257 |
| `r50k_base` | code-davinci-002 | 50,257 |
| `p50k_base` | text-davinci-003 | 50,281 |
| `cl100k_base` | GPT-3.5 / GPT-4 | 100,256 |
| `o200k_base` | GPT-4o | 200,019 |

In [ ]:
import tiktoken

# 对比不同编码器
encoders = {
    'gpt2': tiktoken.get_encoding('gpt2'),
    'cl100k (GPT-4)': tiktoken.get_encoding('cl100k_base'),
    'o200k (GPT-4o)': tiktoken.get_encoding('o200k_base'),
}

test_texts = [
    "Hello, world!",
    "The quick brown fox jumps over the lazy dog.",
    "大语言模型通过预测下一个token来学习",
    "def fibonacci(n):\n    if n <= 1: return n\n    return fibonacci(n-1) + fibonacci(n-2)",
]

for text in test_texts:
    print(f"\n文本: '{text[:50]}{'...' if len(text)>50 else ''}'")
    for name, enc in encoders.items():
        tokens = enc.encode(text)
        decoded = [enc.decode([t]) for t in tokens]
        print(f"  {name:20s}: {len(tokens):3d} tokens  {decoded[:8]}{'...' if len(decoded)>8 else ''}")

print("\n→ 注意中文在不同编码器下的 token 数差异：词表越大 → 中文效率越高")

In [ ]:
# 深入分析：中文 token 效率问题
enc_gpt2 = tiktoken.get_encoding('gpt2')
enc_cl100k = tiktoken.get_encoding('cl100k_base')

chinese = "注意力机制是深度学习的核心技术"
english = "Attention mechanism is the core of deep learning"

cn_gpt2 = enc_gpt2.encode(chinese)
cn_cl100k = enc_cl100k.encode(chinese)
en_gpt2 = enc_gpt2.encode(english)
en_cl100k = enc_cl100k.encode(english)

print("中文效率分析:")
print(f"  '{chinese}'")
print(f"  字符数: {len(chinese)}")
print(f"  GPT-2:    {len(cn_gpt2):3d} tokens (每字符 {len(cn_gpt2)/len(chinese):.1f} tokens)")
print(f"  cl100k:   {len(cn_cl100k):3d} tokens (每字符 {len(cn_cl100k)/len(chinese):.1f} tokens)")

print(f"\n英文效率分析:")
print(f"  '{english}'")
print(f"  单词数: {len(english.split())}")
print(f"  GPT-2:    {len(en_gpt2):3d} tokens")
print(f"  cl100k:   {len(en_cl100k):3d} tokens")

print(f"\n→ GPT-2 tokenizer 对中文效率极差 (UTF-8 byte fallback)")
print(f"→ cl100k 增加了大量中文 token，效率大幅提升")
print(f"→ 这就是为什么 Chinese-LLaMA 需要扩展词表 (Week 7)")

## 五、Embedding 层设计

Tokenizer 将文本转为整数序列，Embedding 层再将整数转为稠密向量 —— 这是模型真正处理的输入。

### 5.1 Embedding 的本质

Embedding 就是一个**查表操作**：

$$\mathbf{e}_i = \mathbf{E}[i] \in \mathbb{R}^{d_{\text{model}}}$$

其中 $\mathbf{E} \in \mathbb{R}^{V \times d_{\text{model}}}$ 是可学习的参数矩阵。

等价于 one-hot 向量乘以 embedding 矩阵：
$$\mathbf{e}_i = \text{onehot}(i) \cdot \mathbf{E}$$

但实际实现直接按索引取行，避免了稀疏矩阵乘法。

### 5.2 主流模型的 Embedding 配置

| 模型 | 词表大小 V | 嵌入维度 d_model | Embedding 参数量 | 占总参数比 |
|------|----------|----------------|-----------------|----------|
| GPT-2 Small | 50,257 | 768 | 38.6M | 31% |
| GPT-2 XL | 50,257 | 1,600 | 80.4M | 5.2% |
| LLaMA-7B | 32,000 | 4,096 | 131M | 1.9% |
| LLaMA-3 8B | 128,256 | 4,096 | 525M | 6.6% |
| LLaMA-70B | 32,000 | 8,192 | 262M | 0.4% |

**规律**：模型越大，Embedding 占总参数的比例越小 —— 真正的「智能」在 Transformer 层中。

In [ ]:
import torch
import torch.nn as nn


class TokenEmbedding(nn.Module):
    """Token Embedding 层，含可选的权重共享。"""

    def __init__(self, vocab_size: int, d_model: int, tie_weights: bool = True):
        super().__init__()
        self.d_model = d_model
        self.embedding = nn.Embedding(vocab_size, d_model)

        self.output_proj = nn.Linear(d_model, vocab_size, bias=False)
        if tie_weights:
            self.output_proj.weight = self.embedding.weight

    def encode(self, token_ids: torch.Tensor) -> torch.Tensor:
        """token_ids → 嵌入向量, 并乘以 sqrt(d_model) 缩放。"""
        return self.embedding(token_ids) * (self.d_model ** 0.5)

    def decode(self, hidden: torch.Tensor) -> torch.Tensor:
        """隐藏状态 → logits (词表上的分数)。"""
        return self.output_proj(hidden)


# 演示
vocab_size, d_model = 32000, 4096
embed = TokenEmbedding(vocab_size, d_model, tie_weights=True)

input_ids = torch.tensor([[1, 234, 5678, 910, 2]])
embeddings = embed.encode(input_ids)
logits = embed.decode(embeddings)

print(f"词表大小: {vocab_size}")
print(f"嵌入维度: {d_model}")
print(f"Embedding 参数量: {vocab_size * d_model / 1e6:.1f}M")
print(f"\n输入 token_ids: {input_ids.shape}")
print(f"嵌入输出:       {embeddings.shape}")
print(f"Logits 输出:    {logits.shape}")

print(f"\n权重共享验证:")
print(f"  embedding.weight 与 output_proj.weight 相同: {embed.embedding.weight is embed.output_proj.weight}")
print(f"  → 节省了 {vocab_size * d_model / 1e6:.1f}M 参数!")

### 5.3 权重共享（Weight Tying）的直觉

为什么 embedding 矩阵可以同时做「输入编码」和「输出解码」？

- **输入端**：`E[token_id]` 把 token 映射到语义空间
- **输出端**：`hidden @ E.T` 计算隐藏状态与每个 token 嵌入的相似度

两端使用同一个矩阵意味着：**如果一个 token 的嵌入与当前隐藏状态最相似，它就是最可能的下一个 token**。

这在语义上是合理的 —— 输入和输出共享同一个语义空间。

注意：并非所有模型都做权重共享。
- LLaMA-1/2: **不共享**（input embedding 和 output head 是独立参数）
- GPT-2: **共享**
- LLaMA-3: **不共享**（因为词表扩大到 128K，共享反而会限制表达能力）

## 六、词表扩展实践（中文适配铺垫）

当一个英文为主的模型需要处理中文时，Byte-level BPE 虽然不会 OOV，但效率极低 —— 一个中文字可能需要 3~4 个 byte token。

**解决方案：词表扩展（Vocabulary Extension）**

核心步骤：
1. 在目标语言语料上训练新的 tokenizer
2. 选取高频且原词表中不存在的 token
3. 添加到原词表并扩展 embedding 矩阵
4. 新 token 的 embedding 需要合理初始化

In [ ]:
def extend_embedding(
    old_embedding: nn.Embedding,
    num_new_tokens: int,
    init_strategy: str = 'mean',
) -> nn.Embedding:
    """
    扩展 Embedding 层以支持新增的 token。
    
    Args:
        old_embedding: 原始 embedding 层
        num_new_tokens: 新增 token 数量
        init_strategy: 初始化策略
            - 'mean': 用已有 embedding 的均值初始化（推荐）
            - 'random': 随机初始化（标准差与已有 embedding 一致）
    """
    old_num, d_model = old_embedding.weight.shape
    new_num = old_num + num_new_tokens
    
    new_embedding = nn.Embedding(new_num, d_model)
    
    with torch.no_grad():
        # 复制原始权重
        new_embedding.weight[:old_num] = old_embedding.weight
        
        if init_strategy == 'mean':
            mean_embed = old_embedding.weight.mean(dim=0)
            new_embedding.weight[old_num:] = mean_embed.unsqueeze(0).expand(num_new_tokens, -1)
        elif init_strategy == 'random':
            std = old_embedding.weight.std().item()
            new_embedding.weight[old_num:] = torch.randn(num_new_tokens, d_model) * std
    
    return new_embedding


# 模拟 Chinese-LLaMA 的词表扩展
original_embed = nn.Embedding(32000, 4096)  # LLaMA 原始
nn.init.normal_(original_embed.weight, std=0.02)

chinese_tokens = 10000  # 新增 10K 中文 token
extended_embed = extend_embedding(original_embed, chinese_tokens, init_strategy='mean')

print(f"原始词表: {original_embed.weight.shape[0]:,} tokens")
print(f"新增中文: {chinese_tokens:,} tokens")
print(f"扩展后:   {extended_embed.weight.shape[0]:,} tokens")
print(f"\nEmbedding 参数量变化: {original_embed.weight.numel()/1e6:.1f}M → {extended_embed.weight.numel()/1e6:.1f}M")
print(f"新增参数: {chinese_tokens * 4096 / 1e6:.1f}M")

# 验证原始权重未被改变
print(f"\n原始权重保留: {torch.allclose(original_embed.weight, extended_embed.weight[:32000])}")

## 七、练习题

### 练习 1：BPE vs Unigram 对比
对同一段中英混合文本，分别用 SentencePiece 的 BPE 和 Unigram 模式训练，比较分词结果有何不同。

### 练习 2：tiktoken 效率分析
分别对 500 字的中文、500 词的英文、500 行的 Python 代码用 `cl100k_base` 编码，比较 token 数量与 token/字符 比。

### 练习 3：Embedding 初始化对比
分别用 `mean` 和 `random` 策略初始化新增 token，计算新 token embedding 与原始 embedding 的余弦相似度分布，思考哪种更好。

### 练习 4（思考题）
为什么 LLaMA-3 把词表从 32K 扩大到 128K？这对训练和推理各有什么影响？